In [1]:
%cd ../

/u02/thanh/workplace/plasma


In [2]:
import inspect
import plasma.data_model as pdm

from pydantic import BaseModel
from typing import NamedTuple, dataclass_transform

from plasma.data_model.base_model.field import Field, Composite, List

In [3]:
@pdm.model
class A:
    a:int
    b:int
    c:list[str]

In [4]:
@pdm.model
class B:
    b1:A
    b2:list[A]
    b3:list

In [5]:
@pdm.model
class C:
    c1:B
    c2:list[B]

In [6]:
pdm.schema(C)

C
  |->c1:B
    |->b1:A
      |->a:int
      |->b:int
      |->c:list[str]
    |->b2:list[A]
      |->@idx:A
        |->a:int
        |->b:int
        |->c:list[str]
    |->b3:list
  |->c2:list[B]
    |->@idx:B
      |->b1:A
        |->a:int
        |->b:int
        |->c:list[str]
      |->b2:list[A]
        |->@idx:A
          |->a:int
          |->b:int
          |->c:list[str]
      |->b3:list

In [7]:
c = C(
        B(
            A(5, 6, ['a', 'b']),
            [
                A(5, 6, ['a', 'b']),
            ],
            [1, 2]
        ),
        [
            B(
                A(5, 6, ['a', 'b']),
                [],
                [1, 3]
            ),
            B(
                A(5, 6, ['a', 'b']),
                [
                    A(7, 8, ['a', 'b']),
                ],
                [1, 3]
            ),
        ]
    )

c

C
  |-c1=B
    |-b1=A
      |-a=5
      |-b=6
      |-c=['a', 'b']
    |-b2:
      A
        |-a=5
        |-b=6
        |-c=['a', 'b']
    |-b3=[1, 2]
  |-c2:
    B
      |-b1=A
        |-a=5
        |-b=6
        |-c=['a', 'b']
      |-b2=[]
      |-b3=[1, 3]
    B
      |-b1=A
        |-a=5
        |-b=6
        |-c=['a', 'b']
      |-b2:
        A
          |-a=7
          |-b=8
          |-c=['a', 'b']
      |-b3=[1, 3]

In [8]:
accessors = pdm.Serializer(C).to_accessors(c)
accessors, len(accessors)

({'c1.b1.a': 5,
  'c1.b1.b': 6,
  'c1.b1.c': ['a', 'b'],
  'c1.b2.0.a': 5,
  'c1.b2.0.b': 6,
  'c1.b2.0.c': ['a', 'b'],
  'c1.b3': [1, 2],
  'c2.0.b1.a': 5,
  'c2.0.b1.b': 6,
  'c2.0.b1.c': ['a', 'b'],
  'c2.0.b2': [],
  'c2.0.b3': [1, 3],
  'c2.1.b1.a': 5,
  'c2.1.b1.b': 6,
  'c2.1.b1.c': ['a', 'b'],
  'c2.1.b2.0.a': 7,
  'c2.1.b2.0.b': 8,
  'c2.1.b2.0.c': ['a', 'b'],
  'c2.1.b3': [1, 3]},
 19)

In [9]:
pdm.Parser(C).from_accessors(accessors)

C
  |-c1=B
    |-b1=A
      |-a=5
      |-b=6
      |-c=['a', 'b']
    |-b2:
      A
        |-a=5
        |-b=6
        |-c=['a', 'b']
    |-b3=[1, 2]
  |-c2:
    B
      |-b1=A
        |-a=5
        |-b=6
        |-c=['a', 'b']
      |-b2=[]
      |-b3=[1, 3]
    B
      |-b1=A
        |-a=5
        |-b=6
        |-c=['a', 'b']
      |-b2:
        A
          |-a=7
          |-b=8
          |-c=['a', 'b']
      |-b3=[1, 3]

In [10]:
struct = pdm.Serializer(C).to_struct(c)
struct

{'c1': {'b1': {'a': 5, 'b': 6, 'c': ['a', 'b']},
  'b2': [{'a': 5, 'b': 6, 'c': ['a', 'b']}],
  'b3': [1, 2]},
 'c2': [{'b1': {'a': 5, 'b': 6, 'c': ['a', 'b']}, 'b2': [], 'b3': [1, 3]},
  {'b1': {'a': 5, 'b': 6, 'c': ['a', 'b']},
   'b2': [{'a': 7, 'b': 8, 'c': ['a', 'b']}],
   'b3': [1, 3]}]}

In [11]:
pdm.Parser(C).from_struct(struct)

C
  |-c1=B
    |-b1=A
      |-a=5
      |-b=6
      |-c=['a', 'b']
    |-b2:
      A
        |-a=5
        |-b=6
        |-c=['a', 'b']
    |-b3=[1, 2]
  |-c2:
    B
      |-b1=A
        |-a=5
        |-b=6
        |-c=['a', 'b']
      |-b2=[]
      |-b3=[1, 3]
    B
      |-b1=A
        |-a=5
        |-b=6
        |-c=['a', 'b']
      |-b2:
        A
          |-a=7
          |-b=8
          |-c=['a', 'b']
      |-b3=[1, 3]

In [12]:
pdm.Serializer(B).to_struct(B(None, [], None))

{'b1': None, 'b2': [], 'b3': None}

In [13]:
pdm.Validator(B)(B(None, [A(None, None, 8)], None))

AttributeError: ('b2.0.c', 'b3')